# 🌌 APEX — Developer Portal

> **Adaptive Platform for Unified AI Configuration, Orchestration and Workspace Management**
>
> *Configure once. Run any supported AI. Manage everything from one unified workspace.*

---

| | |
|---|---|
| **Repository** | [github.com/Nikhil-Kumar-Shah/APEX](https://github.com/Nikhil-Kumar-Shah/APEX) |
| **Author** | Nikhil Kumar Shah |
| **License** | MIT |
| **Entry point** | `notebook/APEX.ipynb` (this file) |
| **Documentation** | `docs/` · `README.md` |

---

### 🗂️ Portal Sections

| # | Section | Purpose |
|---|---|---|
| 0 | 🌟 Welcome | System info, onboarding guide, welcome screen |
| 1 | 📚 Documentation Center | Browse and search all project documentation |
| 2 | ⚙️ Configuration | Runtime and model settings |
| 3 | 🚀 Initialize Runtime | Boot the APEX runtime and live dashboard |
| 4 | 📥 Model Management | Download and load AI models into VRAM |
| 5 | 🌐 API Server | Start the OpenAI-compatible API server |
| 6 | 🗂️ Workspace | Manage project workspaces |
| 7 | 🔬 Diagnostics | Health checks, GPU status, git info |
| 8 | 🛑 Shutdown | Graceful shutdown |

> ⚡ **Run cells sequentially** for first-time setup.
> After initialization, each section can be run independently.

---
## 🌟 Section 0 — Welcome

Run this cell to:
- Mount Google Drive (Colab) for persistent storage
- Clone or update the APEX repository from GitHub
- Configure `sys.path` so all runtime modules are importable
- Display the APEX welcome screen and project information panel

In [ ]:
import sys
import subprocess
from pathlib import Path

# ── 1. Environment Detection ─────────────────────────────────────────────
is_colab = "google.colab" in sys.modules
if is_colab:
    print("[INFO] Google Colab detected.")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("[SUCCESS] Google Drive mounted.")
    except Exception as e:
        print(f"[WARNING] Drive mount failed: {e}. Using temporary storage.")
else:
    print("[INFO] Local environment detected.")

# ── 2. Locate / Fetch Repository ─────────────────────────────────────────
REPO_URL   = "https://github.com/Nikhil-Kumar-Shah/APEX.git"
target_dir = Path("/content/APEX") if is_colab else Path.cwd().parent

if not (target_dir / ".git").exists():
    print("\n[INFO] Cloning APEX repository...")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "-b", "main", REPO_URL, str(target_dir)], check=True)
    print("[SUCCESS] Repository cloned.")
else:
    print("\n[INFO] Updating APEX repository...")
    subprocess.run(["git", "-C", str(target_dir), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(target_dir), "reset", "--hard", "origin/main"], check=True)
    subprocess.run(["git", "-C", str(target_dir), "clean", "-fd"], check=True)
    print("[SUCCESS] Repository updated.")

# ── 3. Configure sys.path ────────────────────────────────────────────────
if str(target_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(target_dir.resolve()))

# Clear stale module cache on kernel restarts
for _mod in list(sys.modules.keys()):
    if _mod.startswith("runtime"):
        del sys.modules[_mod]

# ── 4. Render Welcome Screen ─────────────────────────────────────────────
from runtime.ui.docs_center import DocumentationCenter
dc = DocumentationCenter(root_path=target_dir)
dc.render_welcome()
dc.render_system_info()
dc.render_quick_actions()
print("\n[SUCCESS] Environment ready. Proceed to Section 1 — Documentation Center.")

---
## 📚 Section 1 — Documentation Center

Interactive documentation browser. Navigate and search the entire APEX documentation set without leaving the notebook.

- **Navigation panel** — click any document to display it
- **Search bar** — type to search across all docs instantly
- **Auto-discovery** — new files added to `docs/` appear automatically
- **Source of truth** — content is always read from the Markdown files in the repository

In [ ]:
# ── Documentation Center ─────────────────────────────────────────────────
# Renders the interactive documentation browser.
# Requires Section 0 to have been run (sets up sys.path and `dc`).

# Show onboarding progress (update `completed` list as you run each section)
dc.render_onboarding_steps(completed=[1, 2])

# Render the full interactive Documentation Center
dc.render()

---
## ⚙️ Section 2 — Configuration

Edit runtime settings here before running Section 3.
See [`docs/CONFIGURATION.md`](../docs/CONFIGURATION.md) for the full configuration reference.

| Variable | Default | Description |
|---|---|---|
| `MODEL_ID` | `Qwen/Qwen2.5-0.5B` | Hugging Face model ID |
| `PRECISION` | `float16` | Tensor dtype: `float32` / `float16` / `bfloat16` |
| `API_ENABLED` | `True` | Whether to start the API server |
| `AUTHENTICATION` | `False` | Require Bearer token on all API requests |
| `TRANSPORT` | `cloudflare` | Public tunnel: `cloudflare` / `ngrok` / `local` |

In [ ]:
# ── Runtime Configuration ────────────────────────────────────────────────
# Edit these variables, then run Section 3.

# Model
MODEL_ID   = "Qwen/Qwen2.5-0.5B"   # Hugging Face model ID
PRECISION  = "float16"              # float32 | float16 | bfloat16

# API
API_ENABLED    = True
AUTHENTICATION = False               # True = require Bearer token
TRANSPORT      = "cloudflare"        # cloudflare | ngrok | local

# Cache
CACHE_ENABLED  = True

# ── Print summary ────────────────────────────────────────────────────────
print(f"[CONFIG] Model      : {MODEL_ID}")
print(f"[CONFIG] Precision  : {PRECISION}")
print(f"[CONFIG] API        : {'Enabled' if API_ENABLED else 'Disabled'}")
print(f"[CONFIG] Auth       : {'Enabled' if AUTHENTICATION else 'Disabled'}")
print(f"[CONFIG] Transport  : {TRANSPORT}")

---
## 🚀 Section 3 — Initialize Runtime

Boots the complete APEX lifecycle:
1. Reads and validates `apex.config.json`
2. Initialises Model Manager, Health Monitor, Workspace Manager, and Orchestrator
3. Sets up the API subsystem
4. Launches the Live Status Dashboard

> The **Live Status Dashboard** appears below and updates on every `dashboard.refresh()` call.

In [ ]:
# ── Initialize Runtime ───────────────────────────────────────────────────
from runtime.core.lifecycle import RuntimeLifecycle
from runtime.model.manager import ModelManager
from runtime.core.health import HealthMonitor
from runtime.memory.workspace import WorkspaceManager
from runtime.ui.dashboard import RuntimeDashboard
from runtime.orchestrator.orchestrator import RuntimeOrchestrator
from runtime.api.configuration import APIConfig
from runtime.api.state import RuntimeState
from runtime.api.manager import APIManager

print("Booting APEX Lifecycle...")
lifecycle = RuntimeLifecycle(workspace_path=target_dir)
lifecycle.startup(interactive=False)

cache_path     = lifecycle.drive_manager.get_path("cache")
workspaces_dir = lifecycle.drive_manager.persistence_root / "workspaces"

# Initialise managers
model_manager     = ModelManager(cache_path)
health_monitor    = HealthMonitor(cache_path, model_manager)
workspace_manager = WorkspaceManager(workspaces_dir)
orchestrator      = RuntimeOrchestrator(model_manager, workspace_manager)

workspace_manager.create_workspace("default", "Default Workspace")
workspace_manager.load_workspace("default")

# Setup API subsystem
api_config = APIConfig(
    api_enabled=API_ENABLED,
    transport=TRANSPORT,
    enable_auth=AUTHENTICATION
)
api_state   = RuntimeState()
api_manager = APIManager(api_config, api_state, model_manager)

# Launch Live Status Dashboard
dashboard = RuntimeDashboard()
dashboard.api_manager = api_manager
dashboard.bind_managers(
    lifecycle.config_manager, model_manager, health_monitor, lifecycle, orchestrator
)
dashboard.render()
dashboard.refresh()

---
## 📥 Section 4 — Model Management

### 4a. Download Model
Downloads model weights from Hugging Face Hub.
Weights are cached in your Google Drive — subsequent runs skip the download.

In [ ]:
# ── Download Model ───────────────────────────────────────────────────────
orchestrator.state_machine.transition_to("DOWNLOADING_MODEL")
dashboard.refresh()

model_manager.download_model(MODEL_ID)

orchestrator.state_machine.transition_to("READY")
dashboard.refresh()
print(f"[SUCCESS] {MODEL_ID} downloaded and cached.")

### 4b. Load Model
Allocates model weights into VRAM. Check the dashboard for memory usage.

In [ ]:
# ── Load Model into VRAM ─────────────────────────────────────────────────
orchestrator.state_machine.transition_to("LOADING_MODEL")
dashboard.refresh()

model_manager.load_model(MODEL_ID, precision=PRECISION)
api_state.model_loaded = MODEL_ID

orchestrator.state_machine.transition_to("READY")
dashboard.refresh()
print(f"[SUCCESS] {MODEL_ID} loaded into VRAM.")

---
## 🌐 Section 5 — API Server

Starts the APEX OpenAI-compatible API server.
After startup, the public URL is printed and the dashboard updates.

**Compatible clients:** Continue · Cline · Aider · Open WebUI · LangChain · LlamaIndex · any OpenAI SDK

See [`docs/API_REFERENCE.md`](../docs/API_REFERENCE.md) for the full endpoint catalogue.

In [ ]:
# ── Start API Server ─────────────────────────────────────────────────────
api_manager.start()
dashboard.refresh()

---
## 🗂️ Section 6 — Workspace

APEX workspaces isolate project state — conversation history, symbol indexes, and configuration are stored per-workspace.

See [`docs/Memory_Workspace_Architecture.md`](../docs/Memory_Workspace_Architecture.md) for the full workspace model.

In [ ]:
# ── Workspace Management ─────────────────────────────────────────────────
# Display current workspace
print(f"Active Workspace : {workspace_manager.active_workspace_id}")

# List all workspaces
if hasattr(workspace_manager, 'list_workspaces'):
    workspaces = workspace_manager.list_workspaces()
    print(f"All Workspaces   : {workspaces}")

# ── Create / switch workspace (uncomment to use) ──────────────────────
# workspace_manager.create_workspace("my-project", "My Project")
# workspace_manager.load_workspace("my-project")
# print(f"Switched to: {workspace_manager.active_workspace_id}")

---
## 🔬 Section 7 — Diagnostics

Generates a comprehensive health report covering:
- Python version and platform
- Git repository status (branch, commit)
- GPU and VRAM usage
- Runtime health and API status

In [ ]:
# ── Diagnostics ──────────────────────────────────────────────────────────
import sys as _sys
import subprocess as _sp
import platform as _platform

SEP = '=' * 58
print(SEP)
print("  APEX DIAGNOSTICS REPORT")
print(SEP)

# System
print(f"\n[System]")
print(f"  Python      : {_sys.version.split()[0]}")
print(f"  Platform    : {_platform.system()} {_platform.release()}")
print(f"  Colab       : {'Yes' if is_colab else 'No'}")

# Git
try:
    _branch = _sp.run(["git", "-C", str(target_dir), "rev-parse", "--abbrev-ref", "HEAD"],
                      capture_output=True, text=True, timeout=5).stdout.strip()
    _commit = _sp.run(["git", "-C", str(target_dir), "rev-parse", "--short", "HEAD"],
                      capture_output=True, text=True, timeout=5).stdout.strip()
    _count  = _sp.run(["git", "-C", str(target_dir), "rev-list", "--count", "HEAD"],
                      capture_output=True, text=True, timeout=5).stdout.strip()
    print(f"\n[Git]")
    print(f"  Branch      : {_branch}")
    print(f"  Commit      : {_commit}")
    print(f"  Total commits: {_count}")
except Exception as _e:
    print(f"\n[Git]  Error: {_e}")

# GPU
try:
    import torch as _torch
    if _torch.cuda.is_available():
        _props = _torch.cuda.get_device_properties(0)
        _used  = (_props.total_memory - _torch.cuda.mem_get_info()[0]) / 1e9
        _total = _props.total_memory / 1e9
        print(f"\n[GPU]")
        print(f"  Device      : {_props.name}")
        print(f"  VRAM        : {_used:.1f} / {_total:.1f} GB")
        print(f"  Compute     : {_props.major}.{_props.minor}")
    else:
        print(f"\n[GPU]  No CUDA device. Running on CPU.")
except ImportError:
    print("\n[GPU]  torch not installed.")

# Health
try:
    _report = health_monitor.generate_report()
    print(f"\n[Health]")
    print(f"  Model       : {_report['model_manager'].get('active_model_id', 'None')}")
    print(f"  API Status  : {api_state.api_status.value}")
    print(f"  Queue       : {api_state.queue_size} pending")
    print(f"  Total Reqs  : {api_state.total_requests}")
except Exception as _e:
    print(f"\n[Health]  {_e}")

# Documentation Center stats
try:
    print(f"\n[Documentation]")
    print(f"  Docs found  : {len(dc._nav_entries)}")
    print(f"  Root path   : {dc.root}")
except Exception:
    pass

print(f'\n{SEP}')

---
## 🛑 Section 8 — Shutdown

Gracefully shuts down the API server, unloads model weights from VRAM, and stops the orchestrator.

> Run this cell before ending your Colab session to ensure clean resource release.

In [ ]:
# ── Graceful Shutdown ────────────────────────────────────────────────────
api_manager.stop()
model_manager.unload_model()
orchestrator.shutdown()
dashboard.refresh()
print("[SUCCESS] APEX shutdown complete. Resources released.")